# 5-Minute Returns: Jump Detection

- Intraday return construction from 5-min IBM price data
- Diurnal (time-of-day) variation estimation
- Bipower variation (BV) as a jump-robust volatility measure
- Jump detection via a local threshold
- Jump vs. diffusive returns and their densities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.io import loadmat
from scipy.stats import gaussian_kde
from datetime import datetime
import pandas as pd

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True

## 2. Helper Functions — Date Conversion

- **`YMDid`** — decodes both date (`YYYYMMDD`) and time (`HHMM`) columns into date numbers.
- **`YMDid1`** — decodes the date column only (time set to midnight).

In [ ]:
def YMDid(DD):
    """
    Convert encoded date/time columns to matplotlib date numbers.
    DD[:,0]: date encoded as YYYYMMDD integer
    DD[:,1]: time encoded as HHMM integer
    """
    DD = np.array(DD)
    yyyy = np.floor(DD[:, 0] / 1e4).astype(int)
    mm   = np.floor((DD[:, 0] - 1e4 * yyyy) / 1e2).astype(int)
    dd   = (DD[:, 0] - yyyy * 1e4 - mm * 1e2).astype(int)
    hr   = np.floor(DD[:, 1] / 100).astype(int)
    nn   = (DD[:, 1] - hr * 100).astype(int)

    return np.array([
        mdates.date2num(datetime(y, m, d, h, minute, 0))
        for y, m, d, h, minute in zip(yyyy, mm, dd, hr, nn)
    ])


def YMDid1(DD):
    """
    Convert encoded date column to matplotlib date numbers (date only, no time).
    DD[:,0]: date encoded as YYYYMMDD integer
    """
    DD = np.array(DD)
    yyyy = np.floor(DD[:, 0] / 1e4).astype(int)
    mm   = np.floor((DD[:, 0] - 1e4 * yyyy) / 1e2).astype(int)
    dd   = (DD[:, 0] - yyyy * 1e4 - mm * 1e2).astype(int)

    return np.array([
        mdates.date2num(datetime(y, m, d, 0, 0, 0))
        for y, m, d in zip(yyyy, mm, dd)
    ])

In [ ]:
df = pd.read_csv('data.csv')
df['datetime'] = pd.to_datetime(df['date'].astype(str) + df['time'].astype(str).str.zfill(4), format='%Y%m%d%H%M')

raw = df[['date', 'time', 'price']].values  # numpy array for the rest of the code
YMD_ret = mdates.date2num(df['datetime'].values)  # directly usable for plotting

print(f"Data shape : {raw.shape}")
print(f"First row  : {raw[0, :]}")
print(f"Last row   : {raw[-1, :]}")

## Construct Intraday Log-Returns

Each trading day contains **n + 1 = 78** price observations → **n = 77** returns.

In [ ]:
n       = 77          # number of intraday return intervals per day
T       = raw.shape[0] // (n + 1)   # number of trading days
delta_n = 1 / n

ret = np.zeros(n * T)
DD  = np.zeros((n * T, 2))

for t in range(1, T + 1):
    id_full = np.arange((t - 1) * (n + 1), t * (n + 1))  # n+1 price obs
    idr     = np.arange((t - 1) * n,        t * n)        # n return slots

    prices     = raw[id_full, 2].astype(float)
    ret[idr]   = 100 * np.diff(np.log(prices))
    DD[idr, :] = raw[id_full[1:], :2]   # timestamp of each return

YMD_ret = YMDid(DD)
T       = len(ret) // n   # confirm T

print(f"Trading days T = {T},  total returns = {len(ret)}")

## Diurnal (Time-of-Day) Variation

The diurnal pattern `tod` captures systematic intraday volatility variation.
It is estimated using adjacent bipower products at each intraday interval and
normalised so that its mean equals 1.

In [ ]:
mBVd = np.zeros(n)

for k in range(1, n):          # k = 1 .. n-1
    IDa = np.arange(k,     (T - 1) * n + k + 1, n)
    IDb = np.arange(k - 1, (T - 1) * n + k,     n)
    A, B = ret[IDa], ret[IDb]
    mBVd[k] = (np.pi / 2) * np.mean(np.abs(A) * np.abs(B))

mBVd[0] = mBVd[1]             # mirror edge (MATLAB: mBVd(1) = mBVd(2))

tod = (mBVd / mBVd.sum()) * (delta_n ** -1)
print(f"mean(tod) = {tod.mean():.6f}   (should be ≈ 1.0)")

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, n + 1), tod, '*k', linewidth=2)
ax.set_title('Diurnal Pattern of IBM')
ax.set_xlabel('Intraday interval n')
ax.set_ylabel('tod')
ax.set_ylim([0, 4])
plt.tight_layout()
plt.show()

## Bipower Variation (BV)

BV is a jump-robust estimator of integrated variance:

$$BV_t = \frac{\pi}{2} \cdot \frac{n}{n-1} \sum_{j=2}^{n} |r_{t,j-1}| \, |r_{t,j}|$$

In [ ]:
BV = np.zeros(T)

for t in range(1, T + 1):
    idx1 = np.arange((t - 1) * n,     t * n - 1)
    idx2 = np.arange((t - 1) * n + 1, t * n)
    BV[t - 1] = (np.pi / 2) * (n / (n - 1)) * np.dot(np.abs(ret[idx1]),
                                                       np.abs(ret[idx2]))

print(f"BV stats — mean: {BV.mean():.4f}, min: {BV.min():.4f}, max: {BV.max():.4f}")

## Jump Detection

A return is classified as a **jump** if it exceeds the local threshold
$u_n = \alpha_J \sqrt{BV_t \cdot tod_j} \cdot \Delta_n^{0.49}$, otherwise it is **diffusive**.

In [ ]:
# Local threshold (Kronecker product replicates BV across intraday intervals)
BV_kron = np.kron(BV, tod)          # shape: (n*T,)
un      = np.sqrt(BV_kron) * delta_n ** 0.49
alpha_J = 4

Jd = np.where(np.abs(ret) >  alpha_J * un)[0]   # jump indices
Jc = np.where(np.abs(ret) <= alpha_J * un)[0]   # diffusive indices

r_d = ret[Jd]
r_c = ret[Jc]

print(f"Jump returns      : {len(r_d):>6}  ({100*len(r_d)/len(ret):.2f}%)")
print(f"Diffusive returns : {len(r_c):>6}  ({100*len(r_c)/len(ret):.2f}%)")

## Plot Jump vs. Diffusive Returns Over Time

In [ ]:
YMD_ret_d = YMD_ret[Jd]
YMD_ret_c = YMD_ret[Jc]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

ax1.plot_date(YMD_ret_d, r_d, '*k', markersize=3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.set_title('Jump Returns of IBM')
ax1.set_ylabel('Return (%)')

ax2.plot_date(YMD_ret_c, r_c, '*k', markersize=1, alpha=0.4)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.set_title('Diffusive Returns of IBM')
ax2.set_ylabel('Return (%)')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Kernel Density Estimation

Non-parametric density of jump and diffusive returns.
Bandwidth is set to match MATLAB's `ksdensity(..., 'width', 0.3)` convention.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Jump returns KDE
bw_d  = 0.3 / r_d.std(ddof=1)
kde_d = gaussian_kde(r_d, bw_method=bw_d)
x_d   = np.linspace(r_d.min() - 1, r_d.max() + 1, 500)
ax1.plot(x_d, kde_d(x_d), 'k', linewidth=2)
ax1.set_title('Density Estimation of Jump Returns of IBM')
ax1.set_xlabel('Return (%)')
ax1.set_ylabel('Density')

# Diffusive returns KDE
bw_c  = 0.3 / r_c.std(ddof=1)
kde_c = gaussian_kde(r_c, bw_method=bw_c)
x_c   = np.linspace(-6, 6, 500)
ax2.plot(x_c, kde_c(x_c), 'k', linewidth=2)
ax2.set_xlim([-6, 6])
ax2.set_title('Density Estimation of Diffusive Returns of IBM')
ax2.set_xlabel('Return (%)')
ax2.set_ylabel('Density')

plt.tight_layout()
plt.show()